# Advanced Pandas Homework Assignment

## Overview

This homework assignment is designed to test your understanding of advanced pandas functionalities. You will work with complex data transformations, multi-index operations, performance optimization, and custom functionality extensions.

## Dataset Description

You will be working with three datasets:

1. A sales transactions dataset
2. A customer information dataset
3. A product information dataset

These datasets are designed to simulate real-world data challenges that require advanced pandas techniques to solve efficiently.

## Tasks

#0. ** Generate Data

In [124]:
import pandas as pd
import numpy as np
import datetime
from faker import Faker
import uuid

In [125]:
# Set random seed for reproducibility
np.random.seed(42)
fake = Faker()
Faker.seed(42)

In [ ]:
# Define constants
num_customers = 1000
num_products = 200
num_transactions = 50000
start_date = datetime.datetime(2020, 1, 1)
end_date = datetime.datetime(2023, 12, 31)
days_range = (end_date - start_date).days

In [127]:
# Product categories and subcategories
categories = ['Electronics', 'Clothing', 'Home', 'Food', 'Beauty']
subcategories = {
    'Electronics': ['Phones', 'Computers', 'Accessories', 'TVs', 'Audio'],
    'Clothing': ['Men', 'Women', 'Children', 'Shoes', 'Accessories'],
    'Home': ['Furniture', 'Kitchen', 'Decor', 'Bedding', 'Bath'],
    'Food': ['Produce', 'Bakery', 'Dairy', 'Meat', 'Beverages'],
    'Beauty': ['Skincare', 'Makeup', 'Haircare', 'Fragrance', 'Bath & Body']
}

In [128]:
# Regions and countries
regions = ['North America', 'Europe', 'Asia', 'South America', 'Africa', 'Oceania']
countries_by_region = {
    'North America': ['USA', 'Canada', 'Mexico'],
    'Europe': ['UK', 'Germany', 'France', 'Italy', 'Spain'],
    'Asia': ['China', 'Japan', 'India', 'South Korea', 'Singapore'],
    'South America': ['Brazil', 'Argentina', 'Colombia', 'Chile', 'Peru'],
    'Africa': ['South Africa', 'Egypt', 'Nigeria', 'Kenya', 'Morocco'],
    'Oceania': ['Australia', 'New Zealand', 'Fiji']
}

In [129]:
# Generate customer data
customer_ids = [str(uuid.uuid4()) for _ in range(num_customers)]
customer_data = []

for customer_id in customer_ids:
    region = np.random.choice(regions)
    country = np.random.choice(countries_by_region[region])
    join_date = start_date + datetime.timedelta(days=np.random.randint(0, days_range))

    customer_data.append({
        'customer_id': customer_id,
        'name': fake.name(),
        'email': fake.email(),
        'phone': fake.phone_number(),
        'region': region,
        'country': country,
        'city': fake.city(),
        'join_date': join_date,
        'tier': np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], p=[0.5, 0.3, 0.15, 0.05]),
        'is_active': np.random.choice([True, False], p=[0.9, 0.1])
    })

customers_df = pd.DataFrame(customer_data)


In [130]:
customers_df.head

<bound method NDFrame.head of                               customer_id              name  \
0    497a791d-c477-4b50-a74c-d046e8a3dc1b      Allison Hill   
1    4b8d8cbb-6e4f-499d-a0d9-8aa2219e194b        Sean Blake   
2    a87ee588-4e48-4114-9d6c-749e4c8c0134     Edward Fuller   
3    c55529c9-e062-4272-8923-a11bcd3de873     Melinda Jones   
4    3764bb99-3a58-4687-b5ae-cd8f24c26ca2     Charles Mcgee   
..                                    ...               ...   
995  ee0708f3-5d51-493d-a99b-234d1a593bea  Christine Dillon   
996  743b2364-e353-4359-a382-636735725568      Steven Mills   
997  9df67bea-097d-4ab9-8e76-be29fad3c9c1  Krystal Thompson   
998  560f887d-001e-437f-afcd-875b45bd2a51     Jasmine Scott   
999  37c809f6-8927-4083-8e87-e4b500c412f9    Crystal Baxter   

                         email                phone         region  \
0     donaldgarcia@example.net      +1-219-560-0133  South America   
1    helenpeterson@example.org         651.216.1559         Europe   
2  

In [131]:
# Generate product data
product_ids = [str(uuid.uuid4()) for _ in range(num_products)]
product_data = []

for product_id in product_ids:
    category = np.random.choice(categories)
    subcategory = np.random.choice(subcategories[category])
    launch_date = start_date + datetime.timedelta(days=np.random.randint(0, days_range))

    product_data.append({
        'product_id': product_id,
        'name': fake.word() + ' ' + fake.word().capitalize(),
        'category': category,
        'subcategory': subcategory,
        'price': round(np.random.uniform(10, 1000), 2),
        'cost': round(np.random.uniform(5, 500), 2),
        'weight_kg': round(np.random.uniform(0.1, 20), 2),
        'launch_date': launch_date,
        'is_discontinued': np.random.choice([True, False], p=[0.1, 0.9])
    })

products_df = pd.DataFrame(product_data)


In [132]:
products_df.head()

,product_id,name,category,subcategory,price,cost,weight_kg,launch_date,is_discontinued
0,92304071-ab6c-4d55-9e13-b5576e8ea957,since Between,Food,Produce,838.89,68.05,2.61,2022-10-01,False
1,1b3ccc75-ff88-4635-9cad-d090d95f3d38,finally Doctor,Electronics,Phones,919.21,326.65,2.16,2023-08-26,False
2,50c5a800-7be8-49cd-bd7f-a12d8f2e023d,lose Mouth,Home,Decor,23.50,164.75,11.13,2022-05-14,False
3,9bc46acd-d74b-4543-a04c-7329f511501f,what Necessary,Beauty,Haircare,899.80,278.79,16.26,2020-10-17,False
4,377385ba-8e9b-4d82-ba65-793c3ade4550,everybody Fact,Clothing,Accessories,394.84,123.95,1.99,2022-04-16,False


In [133]:
# Generate transaction data
transaction_data = []

for _ in range(num_transactions):
    transaction_date = start_date + datetime.timedelta(days=np.random.randint(0, days_range))
    customer_id = np.random.choice(customer_ids)

    # Each transaction can have 1-5 items
    num_items = np.random.randint(1, 6)

    for _ in range(num_items):
        product_id = np.random.choice(product_ids)
        product_price = products_df.loc[products_df['product_id'] == product_id, 'price'].iloc[0]
        quantity = np.random.randint(1, 5)

        # Apply random discount
        discount_pct = np.random.choice([0, 0, 0, 0.05, 0.1, 0.15, 0.2], p=[0.6, 0.1, 0.1, 0.05, 0.05, 0.05, 0.05])
        price_after_discount = round(product_price * (1 - discount_pct), 2)

        transaction_id = str(uuid.uuid4())

        transaction_data.append({
            'transaction_id': transaction_id,
            'customer_id': customer_id,
            'product_id': product_id,
            'date': transaction_date,
            'quantity': quantity,
            'unit_price': product_price,
            'discount_pct': discount_pct,
            'price_after_discount': price_after_discount,
            'total_amount': round(quantity * price_after_discount, 2),
            'payment_method': np.random.choice(['Credit Card', 'Debit Card', 'PayPal', 'Cash', 'Bank Transfer']),
            'store_id': np.random.randint(1, 50)
        })

transactions_df = pd.DataFrame(transaction_data)


In [134]:
# Add some missing values and anomalies
transactions_df.loc[np.random.choice(transactions_df.index, size=int(len(transactions_df)*0.01)), 'unit_price'] = np.nan
transactions_df.loc[np.random.choice(transactions_df.index, size=int(len(transactions_df)*0.01)), 'discount_pct'] = np.nan
transactions_df.loc[np.random.choice(transactions_df.index, size=int(len(transactions_df)*0.005)), 'total_amount'] = -1
customers_df.loc[np.random.choice(customers_df.index, size=int(len(customers_df)*0.02)), 'email'] = np.nan
products_df.loc[np.random.choice(products_df.index, size=int(len(products_df)*0.01)), 'price'] = np.nan

# Add some duplicates
dupe_indices = np.random.choice(transactions_df.index, size=int(len(transactions_df)*0.005))
dupes = transactions_df.loc[dupe_indices].copy()
transactions_df = pd.concat([transactions_df, dupes], ignore_index=True)

In [135]:
transactions_df.head()

,transaction_id,customer_id,product_id,date,quantity,unit_price,discount_pct,price_after_discount,total_amount,payment_method,store_id
0,188ec94d-8948-4dfe-985b-6e3c1ec6c0f2,5299d9f1-d044-431a-8231-e853ec87cb12,d9bb2773-b52d-4c9d-9f87-0a28c5f37720,2022-04-01,4,392.08,0.00,392.08,1568.32,PayPal,9
1,a9431fbd-cf0f-4493-8e23-84e92975cf6f,5299d9f1-d044-431a-8231-e853ec87cb12,fb672ab5-a12f-47b0-8daa-fa9754412c58,2022-04-01,3,709.89,0.00,709.89,2129.67,Debit Card,5
2,678bc253-6dab-48b3-baee-08432aa0e642,5dc836d0-f318-44ec-93f1-408a77f621fe,c8cd6c2f-e34e-4332-8124-0f3d6d6a808c,2023-11-22,4,941.92,0.00,941.92,3767.68,Cash,12
3,bda84f5b-9922-49a9-9c39-9fed973cfd13,5dc836d0-f318-44ec-93f1-408a77f621fe,87ec4655-29a5-4de9-90db-6a91f96776ce,2023-11-22,3,995.47,0.00,995.47,2986.41,Cash,38
4,84b5f745-f603-4615-bc08-e79063eb3cb3,5dc836d0-f318-44ec-93f1-408a77f621fe,77d5328d-601a-4c00-a159-5a81594a24e3,2023-11-22,4,557.94,0.05,530.04,2120.16,Credit Card,1


In [136]:
# Create a time-series for customer status updates
status_updates = []
for customer_id in customer_ids:
    # Generate 1-5 status updates per customer
    num_updates = np.random.randint(1, 6)
    for _ in range(num_updates):
        update_date = start_date + datetime.timedelta(days=np.random.randint(0, days_range))
        status_updates.append({
            'customer_id': customer_id,
            'update_date': update_date,
            'tier': np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum']),
            'lifetime_value': round(np.random.uniform(0, 10000), 2),
            'credit_score': np.random.randint(300, 851)
        })

status_updates_df = pd.DataFrame(status_updates)
status_updates_df.sort_values('update_date', inplace=True)

In [137]:
status_updates_df.head()

,customer_id,update_date,tier,lifetime_value,credit_score
2481,00a6db65-9c2f-4fee-ab13-786e3fd6dedd,2020-01-01,Platinum,1803.99,776
1183,61352c34-81bb-41d4-9192-38476a9c8de3,2020-01-01,Silver,1499.63,592
573,b840ad13-9198-44ab-9871-b17ab6484cf0,2020-01-01,Bronze,2869.51,748
2045,b8c596bf-bc3c-46fd-a18f-093a3fff0589,2020-01-02,Bronze,2433.03,631
2107,8ac22fbc-9872-420b-b785-6aea9f478618,2020-01-03,Platinum,4536.36,402


In [138]:
# Print sample data
print("Customers sample:")
print(customers_df.head())
print("\nProducts sample:")
print(products_df.head())
print("\nTransactions sample:")
print(transactions_df.head())
print("\nStatus Updates sample:")
print(status_updates_df.head())

Customers sample:
                            customer_id           name  \
0  497a791d-c477-4b50-a74c-d046e8a3dc1b   Allison Hill   
1  4b8d8cbb-6e4f-499d-a0d9-8aa2219e194b     Sean Blake   
2  a87ee588-4e48-4114-9d6c-749e4c8c0134  Edward Fuller   
3  c55529c9-e062-4272-8923-a11bcd3de873  Melinda Jones   
4  3764bb99-3a58-4687-b5ae-cd8f24c26ca2  Charles Mcgee   

                       email               phone         region      country  \
0   donaldgarcia@example.net     +1-219-560-0133  South America         Peru   
1                        NaN        651.216.1559         Europe       France   
2      barbara10@example.net        441.731.6475  South America     Colombia   
3  amandasanchez@example.com  (748)535-0305x6413        Oceania  New Zealand   
4        julie69@example.com   (332)887-1012x269         Europe        Spain   

            city  join_date      tier  is_active  
0  Robinsonshire 2023-07-18    Silver       True  
1    Herrerafurt 2023-05-23    Bronze       True  

In [139]:
# Save to CSV files if needed
customers_df.to_csv('customers.csv', index=False)
products_df.to_csv('products.csv', index=False)
transactions_df.to_csv('transactions.csv', index=False)
status_updates_df.to_csv('status_updates.csv', index=False)

### Task 1: Advanced Data Transformation and Reshaping

### Task 1 Hints:

- For pivoting complex data, consider using `pivot_table` instead of `pivot` to handle duplicate values
- When working with hierarchical indices, `swaplevel` and `sortlevel` can help organize your data
- Remember that `transform` preserves the shape of the original DataFrame and aligns the results, while `apply` reduces each group to a single row

#1. **Pivot and Melt Operations**

   - Convert the sales data from long format to wide format using `pivot` or `pivot_table`
   - Transform the data back to long format using `melt`
   - Create a pivot table that shows monthly sales totals by product category with subtotals

In [176]:
transactions_df['Month'] = transactions_df['date'].dt.to_period('M')
transactions_df_merged = pd.merge(transactions_df, products_df, on='product_id', how='left')
transactions_df_merged.head()
monthly_category_sales_with_subtotals = transactions_df_merged.pivot_table(index='Month',
                                                                    columns='category',
                                                                    values='total_amount',
                                                                    aggfunc='sum',margins=True,
                                                                    margins_name='Total')
monthly_category_sales_with_subtotals.head()

category,Beauty,Clothing,Electronics,Food,Home,Total
Month,,,,,,
2020-01,773487.93,800213.53,642884.19,961390.67,755325.90,3933302.22
2020-02,632960.54,803318.78,576248.73,853723.21,688720.04,3554971.30
2020-03,819749.87,899412.48,760638.06,938639.73,801132.02,4219572.16
2020-04,727037.46,864801.97,589828.44,935140.56,810239.19,3927047.62
2020-05,737967.11,935772.53,628004.04,951226.76,889221.77,4142192.21


In [178]:
transactions_df_merged_right = pd.merge(transactions_df, products_df, on='product_id', how='right')
transactions_df_merged_right.head()
monthly_category_sales_with_subtotals_right = transactions_df_merged_right.pivot_table(index='Month',
                                                                    columns='category',
                                                                    values='total_amount',
                                                                    aggfunc='sum',margins=True,
                                                                    margins_name='Total')
monthly_category_sales_with_subtotals_right.head()

category,Beauty,Clothing,Electronics,Food,Home,Total
Month,,,,,,
2020-01,773487.93,800213.53,642884.19,961390.67,755325.90,3933302.22
2020-02,632960.54,803318.78,576248.73,853723.21,688720.04,3554971.30
2020-03,819749.87,899412.48,760638.06,938639.73,801132.02,4219572.16
2020-04,727037.46,864801.97,589828.44,935140.56,810239.19,3927047.62
2020-05,737967.11,935772.53,628004.04,951226.76,889221.77,4142192.21


In [141]:
melted_pt = pd.melt(monthly_category_sales_with_subtotals.reset_index(), id_vars=['Month'],var_name='category', value_name='total_amount')
melted_pt.head()

,Month,category,total_amount
0,2020-01,Beauty,773487.93
1,2020-02,Beauty,632960.54
2,2020-03,Beauty,819749.87
3,2020-04,Beauty,727037.46
4,2020-05,Beauty,737967.11


#2. **Multi-level Indexing**

   - Create a hierarchical index on the sales data using customer region, product category, and date
   - Perform operations on specific levels of the hierarchy using `xs`
   - Unstacking and restacking levels to reshape the data for different analyses

In [142]:
### - When working with hierarchical indices, `swaplevel` and `sortlevel` can help organize your data
transactions_product_customer_df_merged = pd.merge(transactions_df_merged, customers_df, on='customer_id', how='left')
transactions_multi_index = transactions_product_customer_df_merged.set_index(['region', 'category', 'Month'])
transactions_multi_index.head()


transaction_id  \
region        category Month                                           
Africa        Food     2022-04  188ec94d-8948-4dfe-985b-6e3c1ec6c0f2   
                       2022-04  a9431fbd-cf0f-4493-8e23-84e92975cf6f   
South America Clothing 2023-11  678bc253-6dab-48b3-baee-08432aa0e642   
              Food     2023-11  bda84f5b-9922-49a9-9c39-9fed973cfd13   
              Beauty   2023-11  84b5f745-f603-4615-bc08-e79063eb3cb3   

                                                         customer_id  \
region        category Month                                           
Africa        Food     2022-04  5299d9f1-d044-431a-8231-e853ec87cb12   
                       2022-04  5299d9f1-d044-431a-8231-e853ec87cb12   
South America Clothing 2023-11  5dc836d0-f318-44ec-93f1-408a77f621fe   
              Food     2023-11  5dc836d0-f318-44ec-93f1-408a77f621fe   
              Beauty   2023-11  5dc836d0-f318-44ec-93f1-408a77f621fe   

                                                          product_id  \
region        category Month                                           
Africa        Food     2022-04  d9bb2773-b52d-4c9d-9f87-0a28c5f37720   
                       2022-04  fb672ab5-a12f-47b0-8daa-fa9754412c58   
South America Clothing 2023-11  c8cd6c2f-e34e-4332-8124-0f3d6d6a808c   
              Food     2023-11  87ec4655-29a5-4de9-90db-6a91f96776ce   
              Beauty   2023-11  77d5328d-601a-4c00-a159-5a81594a24e3   

                                     date  quantity  unit_price  discount_pct  \
region        category Month                                                    
Africa        Food     2022-04 2022-04-01         4      392.08          0.00   
                       2022-04 2022-04-01         3      709.89          0.00   
South America Clothing 2023-11 2023-11-22         4      941.92          0.00   
              Food     2023-11 2023-11-22         3      995.47          0.00   
              Beauty   2023-11 2023-11-22         4      557.94          0.05   

                                price_after_discount  total_amount  \
region        category Month                                         
Africa        Food     2022-04                392.08       1568.32   
                       2022-04                709.89       2129.67   
South America Clothing 2023-11                941.92       3767.68   
              Food     2023-11                995.47       2986.41   
              Beauty   2023-11                530.04       2120.16   

                               payment_method  ...  launch_date  \
region        category Month                   ...                
Africa        Food     2022-04         PayPal  ...   2023-03-20   
                       2022-04     Debit Card  ...   2022-11-16   
South America Clothing 2023-11           Cash  ...   2021-09-06   
              Food     2023-11           Cash  ...   2022-04-16   
              Beauty   2023-11    Credit Card  ...   2021-09-06   

                               is_discontinued            name_y  \
region        category Month                                       
Africa        Food     2022-04           False       Terry Estes   
                       2022-04           False       Terry Estes   
South America Clothing 2023-11           False  Phillip Stephens   
              Food     2023-11           False  Phillip Stephens   
              Beauty   2023-11           False  Phillip Stephens   

                                                email               phone  \
region        category Month                                                
Africa        Food     2022-04      dkhan@example.com          3254883586   
                       2022-04      dkhan@example.com          3254883586   
South America Clothing 2023-11  ryanadams@example.net  (596)756-1308x7612   
              Food     2023-11  ryanadams@example.net  (596)756-1308x7612   
              Beauty   2023-11  ryanadams@example.net  (596)756-1308x76

In [143]:
north_region_sales = transactions_multi_index.xs('North America', level='region').sort_index()
north_region_sales.head()

transaction_id  \
category Month                                           
Beauty   2020-01  3f3fcba6-e76c-4955-8428-9afe238829d6   
         2020-01  a087a957-f097-4547-91ea-37e2e3680e29   
         2020-01  9686e5c6-8428-4e2c-bc1f-690c8df7c529   
         2020-01  61c0a9fa-8b60-4701-bc8c-9eb297f493c1   
         2020-01  36533cd1-c0c8-4ebf-a129-434d8f4698b2   

                                           customer_id  \
category Month                                           
Beauty   2020-01  a888ac06-221d-416b-b532-653ace865a02   
         2020-01  5c7bd5c8-e870-4d79-81a8-ee996c7bd4ed   
         2020-01  4c0459f1-659c-4297-8150-f2890c45cbcf   
         2020-01  fd3399fd-6c48-4433-9cd7-70742d013252   
         2020-01  0d726593-5e7d-4f97-9fa0-94bbfbf11ee9   

                                            product_id       date  quantity  \
category Month                                                                
Beauty   2020-01  7949de4a-bc36-43a1-ae3a-fa7028df5d59 2020-01-10         2   
         2020-01  e3b742c2-04be-4308-8d5b-9726214323b0 2020-01-16         1   
         2020-01  076318b5-35b0-47aa-9224-02fd5898f8a2 2020-01-02         1   
         2020-01  7949de4a-bc36-43a1-ae3a-fa7028df5d59 2020-01-18         1   
         2020-01  e8ab3d6f-f28c-4f62-9e1a-c9903be75792 2020-01-31         1   

                  unit_price  discount_pct  price_after_discount  \
category Month                                                     
Beauty   2020-01      510.87           0.0                510.87   
         2020-01      951.31           0.0                951.31   
         2020-01      100.51           0.2                 80.41   
         2020-01      510.87           0.2                408.70   
         2020-01      187.24           0.0                187.24   

                  total_amount payment_method  ...  launch_date  \
category Month                                 ...                
Beauty   2020-01       1021.74         PayPal  ...   2022-05-14   
         2020-01        951.31    Credit Card  ...   2023-07-26   
         2020-01         80.41  Bank Transfer  ...   2023-06-02   
         2020-01        408.70  Bank Transfer  ...   2022-05-14   
         2020-01        187.24    Credit Card  ...   2020-05-27   

                 is_discontinued           name_y                    email  \
category Month                                                               
Beauty   2020-01           False    Jeanette Luna       gina90@example.org   
         2020-01           False    Kenneth Smith    deborah60@example.com   
         2020-01           False      Shane Short      sonia62@example.com   
         2020-01           False  Danielle Warner  brennaneric@example.com   
         2020-01           False    Robert Butler   kanejeremy@example.net   

                                   phone  country              city  \
category Month                                                        
Beauty   2020-01   +1-812-548-2554x42867   Canada     West Margaret   
         2020-01        712.319.8819x277   Mexico    Port Marymouth   
         2020-01  001-354-902-6355x82430   Mexico        Parkerview   
         2020-01            289-338-9846      USA         Gilesview   
         2020-01     +1-506-404-8058x925   Canada  Lake Jessicafurt   

                  join_date    tier is_active  
category Month                                 
Beauty   2020-01 2022-01-14  Silver      True  
         2020-01 2022-09-06  Bronze      True  
         2020-01 2023-01-16  Bronze      True  
         2020-01 2021-09-03  Silver      True  
         2020-01 2023-02-12  Silver      True  

[5 rows x 26 columns]

In [144]:
north_region_sales2 = transactions_multi_index.xs('North America', level='region').swaplevel('category', 'Month').sort_index()
north_region_sales2.head()

transaction_id  \
Month   category                                         
2020-01 Beauty    3f3fcba6-e76c-4955-8428-9afe238829d6   
        Beauty    a087a957-f097-4547-91ea-37e2e3680e29   
        Beauty    9686e5c6-8428-4e2c-bc1f-690c8df7c529   
        Beauty    61c0a9fa-8b60-4701-bc8c-9eb297f493c1   
        Beauty    36533cd1-c0c8-4ebf-a129-434d8f4698b2   

                                           customer_id  \
Month   category                                         
2020-01 Beauty    a888ac06-221d-416b-b532-653ace865a02   
        Beauty    5c7bd5c8-e870-4d79-81a8-ee996c7bd4ed   
        Beauty    4c0459f1-659c-4297-8150-f2890c45cbcf   
        Beauty    fd3399fd-6c48-4433-9cd7-70742d013252   
        Beauty    0d726593-5e7d-4f97-9fa0-94bbfbf11ee9   

                                            product_id       date  quantity  \
Month   category                                                              
2020-01 Beauty    7949de4a-bc36-43a1-ae3a-fa7028df5d59 2020-01-10         2   
        Beauty    e3b742c2-04be-4308-8d5b-9726214323b0 2020-01-16         1   
        Beauty    076318b5-35b0-47aa-9224-02fd5898f8a2 2020-01-02         1   
        Beauty    7949de4a-bc36-43a1-ae3a-fa7028df5d59 2020-01-18         1   
        Beauty    e8ab3d6f-f28c-4f62-9e1a-c9903be75792 2020-01-31         1   

                  unit_price  discount_pct  price_after_discount  \
Month   category                                                   
2020-01 Beauty        510.87           0.0                510.87   
        Beauty        951.31           0.0                951.31   
        Beauty        100.51           0.2                 80.41   
        Beauty        510.87           0.2                408.70   
        Beauty        187.24           0.0                187.24   

                  total_amount payment_method  ...  launch_date  \
Month   category                               ...                
2020-01 Beauty         1021.74         PayPal  ...   2022-05-14   
        Beauty          951.31    Credit Card  ...   2023-07-26   
        Beauty           80.41  Bank Transfer  ...   2023-06-02   
        Beauty          408.70  Bank Transfer  ...   2022-05-14   
        Beauty          187.24    Credit Card  ...   2020-05-27   

                 is_discontinued           name_y                    email  \
Month   category                                                             
2020-01 Beauty             False    Jeanette Luna       gina90@example.org   
        Beauty             False    Kenneth Smith    deborah60@example.com   
        Beauty             False      Shane Short      sonia62@example.com   
        Beauty             False  Danielle Warner  brennaneric@example.com   
        Beauty             False    Robert Butler   kanejeremy@example.net   

                                   phone  country              city  \
Month   category                                                      
2020-01 Beauty     +1-812-548-2554x42867   Canada     West Margaret   
        Beauty          712.319.8819x277   Mexico    Port Marymouth   
        Beauty    001-354-902-6355x82430   Mexico        Parkerview   
        Beauty              289-338-9846      USA         Gilesview   
        Beauty       +1-506-404-8058x925   Canada  Lake Jessicafurt   

                  join_date    tier is_active  
Month   category                               
2020-01 Beauty   2022-01-14  Silver      True  
        Beauty   2022-09-06  Bronze      True  
        Beauty   2023-01-16  Bronze      True  
        Beauty   2021-09-03  Silver      True  
        Beauty   2023-02-12  Silver      True  

[5 rows x 26 columns]

In [145]:
north_region_sales_by_category = north_region_sales.groupby('category')
north_region_sales_by_category.head()

transaction_id  \
category    Month                                           
Beauty      2020-01  3f3fcba6-e76c-4955-8428-9afe238829d6   
            2020-01  a087a957-f097-4547-91ea-37e2e3680e29   
            2020-01  9686e5c6-8428-4e2c-bc1f-690c8df7c529   
            2020-01  61c0a9fa-8b60-4701-bc8c-9eb297f493c1   
            2020-01  36533cd1-c0c8-4ebf-a129-434d8f4698b2   
Clothing    2020-01  51389ebc-181d-4c45-b3af-fd7dd8b6571f   
            2020-01  2578bef7-be98-4cb5-a51b-6876355578be   
            2020-01  ee224b7c-de71-42b0-a487-1cdcf8d2ba5d   
            2020-01  b656b73d-8f31-4fda-be97-f305624d4b7b   
            2020-01  cffbe8f6-9c82-41f9-80b3-68e7d48c2c09   
Electronics 2020-01  28f24da8-d125-43bd-8978-786caa070d3a   
            2020-01  8aedd1d0-80cc-45af-8563-b57e618b4cf3   
            2020-01  71f0ec2d-ca9f-4e4b-bb89-22ad53c7657e   
            2020-01  2d39a79a-ffa9-4574-8b50-42dfb4211324   
            2020-01  23fcc054-025b-41ba-944b-d50d5fb156b2   
Food        2020-01  dd0ec325-a0ee-4ab5-9afe-768b6b926d42   
            2020-01  8cb26137-4cb0-4bf4-a9f3-70c6b0566763   
            2020-01  5581845b-9900-43a4-ba04-bb5ba30d8cae   
            2020-01  37562df8-d914-4c14-8006-34e46d55cb1e   
            2020-01  6dbe4ee2-87d0-44e6-bb43-89e3d94ce51b   
Home        2020-01  8184830d-3395-4e4a-a32d-1584cf60e8dc   
            2020-01  d89ab371-993b-46ae-b9aa-cbd3d53da3cf   
            2020-01  c1624ef7-3a55-426b-a77f-1f3de05f907c   
            2020-01  c834ffa6-6f04-438f-8baf-984c46e15073   
            2020-01  a4d35219-89c7-4b3a-8068-13733fa0e654   

                                              customer_id  \
category    Month                                           
Beauty      2020-01  a888ac06-221d-416b-b532-653ace865a02   
            2020-01  5c7bd5c8-e870-4d79-81a8-ee996c7bd4ed   
            2020-01  4c0459f1-659c-4297-8150-f2890c45cbcf   
            2020-01  fd3399fd-6c48-4433-9cd7-70742d013252   
            2020-01  0d726593-5e7d-4f97-9fa0-94bbfbf11ee9   
Clothing    2020-01  189bbce0-70f0-4506-a209-92b7ed48bf3d   
            2020-01  f6366b42-b22f-4ba6-91ab-2b00c39f344e   
            2020-01  0c9c0000-437e-4b2d-8783-5ff5593f8187   
            2020-01  a888ac06-221d-416b-b532-653ace865a02   
            2020-01  a888ac06-221d-416b-b532-653ace865a02   
Electronics 2020-01  a888ac06-221d-416b-b532-653ace865a02   
            2020-01  a9c86d51-3e9a-4b07-980b-252bbb3d2af2   
            2020-01  4c0459f1-659c-4297-8150-f2890c45cbcf   
            2020-01  2b9c73ef-c5d5-4b33-ab99-0220b43645ee   
            2020-01  189bbce0-70f0-4506-a209-92b7ed48bf3d   
Food        2020-01  a888ac06-221d-416b-b532-653ace865a02   
            2020-01  a888ac06-221d-416b-b532-653ace865a02   
            2020-01  4c0459f1-659c-4297-8150-f2890c45cbcf   
            2020-01  4c0459f1-659c-4297-8150-f2890c45cbcf   
            2020-01  2b9c73ef-c5d5-4b33-ab99-0220b43645ee   
Home        2020-01  a888ac06-221d-416b-b532-653ace865a02   
            2020-01  e3afa109-a0ac-4116-93d1-30be391eb7d7   
            2020-01  5c7bd5c8-e870-4d79-81a8-ee996c7bd4ed   
            2020-01  d88d2e0b-eb3e-4e65-a68e-4fc434cb2470   
            2020-01  189bbce0-70f0-4506-a209-92b7ed48bf3d   

                                               product_id       date  \
category    Month                                                      
Beauty      2020-01  7949de4a-bc36-43a1-ae3a-fa7028df5d59 2020-01-10   
            2020-01  e3b742c2-04be-4308-8d5b-9726214323b0 2020-01-16   
            2020-01  076318b5-35b0-47aa-9224-02fd5898f8a2 2020-01-02   
            2020-01  7949de4a-bc36-43a1-ae3a-fa7028df5d59 2020-01-18   
            2020-01  e8ab3d6f-f28c-4f62-9e1a-c9903be75792 2020-01-31   
Clothing    2020-01  b5950a43-6abc-4198-beb0-2e62dc044917 2020-01-30   
            2020-01  5cb9d913-7b73-41ae-b6db-da8462b8bfdd 2020-01-04   
            2020-01  688685a8-20e7-4a7e-93df-3a19da52869b 2020-01-28   
            2020-01  6c1119

In [146]:
north_clothing_sales = transactions_multi_index.xs(('North America', 'Clothing'), level=('region', 'category'))
north_clothing_sales.head()

,transaction_id,customer_id,product_id,date,quantity,unit_price,discount_pct,price_after_discount,total_amount,payment_method,...,launch_date,is_discontinued,name_y,email,phone,country,city,join_date,tier,is_active
Month,,,,,,,,,,,,,,,,,,,,,
2022-10,4d15eeff-a652-41d8-bd33-c4974d64ecec,5839206d-7883-4964-9f04-4d79128d4947,191eaa1a-de70-44bc-91bc-6b85de087788,2022-10-22,1,542.16,0.0,542.16,542.16,Bank Transfer,...,2020-11-10,False,Michelle Brock,mchase@example.org,001-627-202-6728x8574,USA,Garciaburgh,2020-03-03,Bronze,True
2021-08,9c15bffd-7a72-483e-bd48-bebe0ceefdca,45cd6d42-d46a-4a51-a68f-c75b430d5ab8,b5950a43-6abc-4198-beb0-2e62dc044917,2021-08-17,1,341.44,0.0,341.44,341.44,Cash,...,2021-09-08,False,Willie Chavez,amy14@example.net,001-385-752-4578x71584,Mexico,Mayside,2023-12-02,Silver,True
2022-02,d36923e9-a530-4e8e-a7da-d47d9d91960a,6f24d44f-c4e0-4d5c-a27b-4061a88ae1c5,59158044-ae10-4741-9291-9b3ab0fa226a,2022-02-23,2,539.62,0.0,539.62,1079.24,PayPal,...,2022-01-31,False,Stacy Weber,zterry@example.net,(334)610-8562x8843,USA,Katherineburgh,2022-07-07,Bronze,True
2020-08,f3f3bfb7-5b1a-42e4-8baa-849aa9d6675d,897aea1d-3010-45b1-a5a5-d4da709984ca,45ee9ab1-905c-4887-af5a-6d20aa60d9ab,2020-08-23,3,812.06,0.0,812.06,2436.18,Credit Card,...,2020-08-08,False,Bailey Clay,martindonald@example.com,4666489008,USA,Michaelville,2023-06-05,Silver,False
2020-05,4287e8d8-8a88-431d-9ddd-e4389ea5b393,3c2d0491-fc47-4ae9-9be8-375118ecefc1,96d7bc8e-a4d7-45ff-83d3-7a80af28db43,2020-05-08,3,992.21,0.0,992.21,2976.63,Bank Transfer,...,2022-12-27,False,James Davis,thomasboyle@example.org,563-274-5499x045,Mexico,South Kurtfurt,2023-05-04,Silver,True


In [147]:
NorthAm_Clothing_202001_sales = transactions_multi_index.xs(('North America', 'Clothing', '2020-01'), level=('region', 'category', 'Month')) 
NorthAm_Clothing_202001_sales.head()

transaction_id  \
region        category Month                                           
North America Clothing 2020-01  51389ebc-181d-4c45-b3af-fd7dd8b6571f   
                       2020-01  2578bef7-be98-4cb5-a51b-6876355578be   
                       2020-01  ee224b7c-de71-42b0-a487-1cdcf8d2ba5d   
                       2020-01  b656b73d-8f31-4fda-be97-f305624d4b7b   
                       2020-01  cffbe8f6-9c82-41f9-80b3-68e7d48c2c09   

                                                         customer_id  \
region        category Month                                           
North America Clothing 2020-01  189bbce0-70f0-4506-a209-92b7ed48bf3d   
                       2020-01  f6366b42-b22f-4ba6-91ab-2b00c39f344e   
                       2020-01  0c9c0000-437e-4b2d-8783-5ff5593f8187   
                       2020-01  a888ac06-221d-416b-b532-653ace865a02   
                       2020-01  a888ac06-221d-416b-b532-653ace865a02   

                                                          product_id  \
region        category Month                                           
North America Clothing 2020-01  b5950a43-6abc-4198-beb0-2e62dc044917   
                       2020-01  5cb9d913-7b73-41ae-b6db-da8462b8bfdd   
                       2020-01  688685a8-20e7-4a7e-93df-3a19da52869b   
                       2020-01  6c111907-467b-4538-b6d3-896ef4972291   
                       2020-01  a9a4378d-40cc-4ea8-816c-0823a4113041   

                                     date  quantity  unit_price  discount_pct  \
region        category Month                                                    
North America Clothing 2020-01 2020-01-30         4      341.44          0.00   
                       2020-01 2020-01-04         3      814.20          0.00   
                       2020-01 2020-01-28         1      794.02          0.00   
                       2020-01 2020-01-20         4      838.06          0.00   
                       2020-01 2020-01-20         2      966.61          0.15   

                                price_after_discount  total_amount  \
region        category Month                                         
North America Clothing 2020-01                341.44       1365.76   
                       2020-01                814.20       2442.60   
                       2020-01                794.02        794.02   
                       2020-01                838.06       3352.24   
                       2020-01                821.62       1643.24   

                               payment_method  ...  launch_date  \
region        category Month                   ...                
North America Clothing 2020-01         PayPal  ...   2021-09-08   
                       2020-01  Bank Transfer  ...   2023-09-03   
                       2020-01  Bank Transfer  ...   2022-04-22   
                       2020-01  Bank Transfer  ...   2020-03-05   
                       2020-01  Bank Transfer  ...   2020-06-25   

                               is_discontinued         name_y  \
region        category Month                                    
North America Clothing 2020-01           False  William Brown   
                       2020-01           False     Cheryl Lee   
                       2020-01           False    Dana Robles   
                       2020-01           False  Jeanette Luna   
                       2020-01           False  Jeanette Luna   

                                                    email  \
region        category Month                                
North America Clothing 2020-01  sarahgonzales@example.com   
                       2020-01   schwartzlisa@example.org   
                       2020-01   hannahcurtis@example.org   
                       2020-01         gina90@example.org   
                       2020-01         gina90@example.org   

                                                phone  country  \
region        category Month                                

In [148]:
print("\n--- Unstacking and Restacking levels to reshape the data ---")

# Unstack 'Product_Category' to make it columns
# This moves the 'Product_Category' level from the index to the columns
try:
    sales_unstacked_category = transactions_multi_index.unstack(level=['region', 'category', 'Month'])
    sales_unstacked = transactions_multi_index.unstack(level=['region', 'category', 'Month'])
    print("\nDataFrame after unstacking 'Product_Category':")
    sales_unstacked_category.head()
except ValueError as e:
    print(f"ValueError: {e}")
    print("\nExplanation: `unstack` expects unique values at the level it's trying to unstack for each combination of the higher levels. Since there are two entries for ('North', 'Electronics') on '2024-01-05', pandas doesn't know which 'Sales_Amount' or 'Product_Name' to place in the new '2024-01-05' column, hence the error.")



--- Unstacking and Restacking levels to reshape the data ---
ValueError: Length mismatch: Expected axis has 150805 elements, new values have 1440 elements

Explanation: `unstack` expects unique values at the level it's trying to unstack for each combination of the higher levels. Since there are two entries for ('North', 'Electronics') on '2024-01-05', pandas doesn't know which 'Sales_Amount' or 'Product_Name' to place in the new '2024-01-05' column, hence the error.


In [149]:
# When you have duplicates at the level you want to unstack (or pivot),
# you should use `pivot_table` to aggregate the values.
# Here, we want to sum the Sales_Amount for duplicate entries.
sales_aggregated_pivot_table = transactions_multi_index.pivot_table(
    index=['region', 'category'], # These will form the new multi-index rows
    columns='date',                               # This will become the new columns
    values='total_amount',                        # The values to aggregate
    aggfunc='sum'                                 # How to handle duplicates (sum them up)
)
print("\nSales data aggregated and unstacked using `pivot_table` (handling duplicates):")
print(sales_aggregated_pivot_table)


Sales data aggregated and unstacked using `pivot_table` (handling duplicates):
date                       2020-01-01  2020-01-02  2020-01-03  2020-01-04  \
region        category                                                      
Africa        Beauty          4374.39     2939.24     8149.46    10484.81   
              Clothing        3370.30     1910.14     9278.20     5335.47   
              Electronics         NaN     2485.78    10311.61     1493.87   
              Food            1331.88     5010.39     3635.36     8071.89   
              Home                NaN     4700.02     6604.78     9163.88   
Asia          Beauty          3534.42    10699.18     5215.34     3734.47   
              Clothing        3013.47     9266.33     2116.36     9796.93   
              Electronics     1812.52     4186.20     2486.47     2642.96   
              Food             296.75    16228.81         NaN    16627.34   
              Home            2199.99     7805.27     1506.63     1382.53

3. **Advanced GroupBy Operations**
   - Use the `transform` method to normalize sales values within groups
   - Apply multiple aggregation functions simultaneously using `agg`
   - Implement custom aggregation functions
   - Use filter operations to select groups meeting specific criteria

In [150]:
# Use the `transform` method to normalize sales values within groups
# Normalize Sales_Amount within each Product_Category
# This calculates the z-score (standard score) for each sales amount relative to its product category.
# It returns a Series with the same index as the original DataFrame.
transactions_product_customer_df_merged['Sales_Amount_Normalized_by_Category'] = transactions_product_customer_df_merged.groupby('category')['total_amount'].transform(lambda x: (x - x.mean()) / x.std())
print("\nSales DataFrame with Sales_Amount normalized within each Product_Category (using transform):")
print(transactions_product_customer_df_merged[['category', 'total_amount', 'Sales_Amount_Normalized_by_Category']])



Sales DataFrame with Sales_Amount normalized within each Product_Category (using transform):
           category  total_amount  Sales_Amount_Normalized_by_Category
0              Food       1568.32                             0.286634
1              Food       2129.67                             0.854352
2          Clothing       3767.68                             2.299147
3              Food       2986.41                             1.720810
4            Beauty       2120.16                             1.089722
...             ...           ...                                  ...
150800       Beauty        234.71                            -0.903191
150801  Electronics        294.24                            -0.909390
150802     Clothing       1588.04                             0.093019
150803         Home        475.29                            -0.890065
150804     Clothing        377.42                            -1.132314

[150805 rows x 3 columns]


In [151]:
# Apply multiple aggregation functions simultaneously using `agg`
# Group by Customer_Region and Product_Category, then calculate mean, sum, and count of Sales_Amount
region_category_sales_summary = transactions_product_customer_df_merged.groupby(['region', 'category']).agg(
    Total_Sales=('total_amount', 'sum'),
    Average_Sales=('total_amount', 'mean'),
    Number_of_Sales=('total_amount', 'count'),
    Min_Sales=('total_amount', 'min'),
    Max_Sales=('total_amount', 'max')
)
print("\nSales Summary by Customer_Region and Product_Category (using agg with multiple functions):")
print(region_category_sales_summary)


Sales Summary by Customer_Region and Product_Category (using agg with multiple functions):
                           Total_Sales  Average_Sales  Number_of_Sales  \
region        category                                                   
Africa        Beauty        5932172.45    1068.090106             5554   
              Clothing      7297847.29    1488.445297             4903   
              Electronics   5423543.08    1146.626444             4730   
              Food          7876309.30    1281.951383             6144   
              Home          6555599.84    1350.556209             4854   
Asia          Beauty        6111817.95    1087.318618             5621   
              Clothing      7395392.89    1498.559856             4935   
              Electronics   5555530.98    1150.928316             4827   
              Food          7969754.41    1280.281833             6225   
              Home          6644991.14    1354.186089             4907   
Europe        Beauty

In [152]:
# Implement custom aggregation functions
# Let's create a custom function to calculate the Interquartile Range (IQR)
def iqr(series):
    return series.quantile(0.75) - series.quantile(0.25)

In [153]:
custom_agg_sales = transactions_product_customer_df_merged.groupby('category').agg(
    Total_Sales=('total_amount', 'sum'),
    IQR_Sales=('total_amount', iqr), # Using the custom IQR function
    Unique_Products=('product_id', lambda x: x.nunique()) # Custom lambda for unique count
)
print("\nSales Summary by Product_Category with custom aggregation (IQR and Unique Products):")
print(custom_agg_sales)


Sales Summary by Product_Category with custom aggregation (IQR and Unique Products):
             Total_Sales  IQR_Sales  Unique_Products
category                                            
Beauty       35023185.90    1356.18               43
Clothing     41749740.21    1450.89               37
Electronics  31298193.49    1230.83               36
Food         45547184.78    1443.09               47
Home         37724534.26    1459.43               37


In [154]:
# Use filter operations to select groups meeting specific criteria
# Filter groups where the total sales for a Product_Category is greater than 5000000
high_sales_categories = transactions_product_customer_df_merged.groupby('category').filter(lambda x: x['total_amount'].sum() > 5000).groupby('category')
print("\nSales records for Product_Categories with total sales > 5000000 (using filter):")
print(high_sales_categories)
# print(high_sales_categories[['category', 'Total_Sales']])



Sales records for Product_Categories with total sales > 5000000 (using filter):


In [155]:
summary_high_sales = high_sales_categories.agg(
    Total_Sales=('total_amount', 'sum'),
    Average_Sales=('total_amount', 'mean'),
    Number_of_Sales=('total_amount', 'count'),
    Min_Sales=('total_amount', 'min'),
    Max_Sales=('total_amount', 'max')
)
print("\nSummary of high sales categories:")
print(summary_high_sales)


Summary of high sales categories:
             Total_Sales  Average_Sales  Number_of_Sales  Min_Sales  Max_Sales
category                                                                      
Beauty       35023185.90    1089.198753            32155       -1.0    3848.00
Clothing     41749740.21    1496.138334            27905       -1.0    3968.84
Electronics  31298193.49    1144.483618            27347       -1.0    3936.76
Food         45547184.78    1284.901399            35448       -1.0    3981.88
Home         37724534.26    1349.715000            27950       -1.0    3982.72


In [156]:
# Filter groups where a Customer_Region has more than 3 sales records
active_regions = transactions_product_customer_df_merged.groupby('region').filter(lambda x: len(x) > 3).groupby('region')
print("\nSales records for Customer_Regions with more than 3 sales (using filter):")
print(active_regions)


Sales records for Customer_Regions with more than 3 sales (using filter):


In [157]:
summary_active_regions = active_regions.agg(
    Total_Sales=('total_amount', 'sum'),
    Average_Sales=('total_amount', 'mean'),
    Number_of_Sales=('total_amount', 'count'),
    Min_Sales=('total_amount', 'min'),
    Max_Sales=('total_amount', 'max')
)
print("\nSummary of active regions:")
print(summary_active_regions)


Summary of active regions:
               Total_Sales  Average_Sales  Number_of_Sales  Min_Sales  \
region                                                                  
Africa         33085471.96    1263.527667            26185       -1.0   
Asia           33677487.37    1270.129639            26515       -1.0   
Europe         30525134.50    1266.287833            24106       -1.0   
North America  26718240.29    1254.907721            21291       -1.0   
Oceania        33604073.44    1282.304565            26206       -1.0   
South America  33732431.08    1272.825865            26502       -1.0   

               Max_Sales  
region                    
Africa           3982.72  
Asia             3982.72  
Europe           3982.72  
North America    3982.72  
Oceania          3982.72  
South America    3982.72  


### Task 2: Advanced Merging and Joining


### Task 2 Hints:

- When performing complex joins, consider executing them in stages rather than all at once
- For time-based joins, make sure to sort your data by time first
- The `pd.merge_asof` function requires sorted data on the join keys
- Consider using `indicator=True` in merge operations to track the source of each row

1. **Complex Joins**

   - Perform a three-way join between sales, customer, and product datasets
   - Implement a self-join on the customer dataset to identify hierarchical relationships
   - Use different join types (left, right, inner, outer) and compare the results

In [170]:
def join_df_by(df1, df2, on, how='inner'):
    """
    Joins two DataFrames on specified columns with error handling.
    
    Parameters:
    - df1: First DataFrame
    - df2: Second DataFrame
    - on: Column(s) to join on
    - how: Type of join ('inner', 'outer', 'left', 'right')
    
    Returns:
    - Merged DataFrame or None if an error occurs
    """
    try:
        return pd.merge(df1, df2, on=on, how=how)
    except Exception as e:
        print(f"Error joining DataFrames: {e}")
        return None

In [200]:
def show_df_info(df, name, columns=None):
    """
    Prints basic information about a DataFrame.
    
    Parameters:
    - df: DataFrame to analyze
    - name: Name of the DataFrame for display purposes
    """
    print(f"\n{name} DataFrame Info:")
    print(f"Shape: {df.shape}")
    if columns != None:
        df_to_display = df
        if len(columns) > 0:
            print(f"Columns: {columns}")
            df_to_display = df[columns]
        print("Description:")
        print(df_to_display.describe(include="all"))
    
    

In [204]:
#Transactions with Customers and Products
all_joints_df= {}
tran = "transactions"
cust = "customers"
prod = "products"
join_dim1 = "customer_id"
joint_types = ['left', 'right', 'inner', 'outer']
for jt in joint_types:
    name1 = f"{tran}_{cust}_{jt}"
    name2 = f"{tran}_{cust}_{prod}_{jt}"
    all_joints_df[name1] = join_df_by(transactions_df, customers_df, on=join_dim1, how=jt)
    all_joints_df[name2] = join_df_by(all_joints_df[name1], products_df, on='product_id', how=jt)
    
empty_cols = []
cols = ["region", "total_amount"]
used_cols = cols
    
# Show info for each joined DataFrame
for name, df in all_joints_df.items():
    show_df_info(df, name, used_cols)
    




transactions_customers_left DataFrame Info:
Shape: (150805, 21)
Columns: ['region', 'total_amount']
Description:
        region   total_amount
count   150805  150805.000000
unique       6            NaN
top       Asia            NaN
freq     26515            NaN
mean       NaN    1268.809646
std        NaN     979.377482
min        NaN      -1.000000
25%        NaN     487.780000
50%        NaN     962.000000
75%        NaN    1898.200000
max        NaN    3982.720000

transactions_customers_products_left DataFrame Info:
Shape: (150805, 29)
Columns: ['region', 'total_amount']
Description:
        region   total_amount
count   150805  150805.000000
unique       6            NaN
top       Asia            NaN
freq     26515            NaN
mean       NaN    1268.809646
std        NaN     979.377482
min        NaN      -1.000000
25%        NaN     487.780000
50%        NaN     962.000000
75%        NaN    1898.200000
max        NaN    3982.720000

transactions_customers_right DataFrame Inf

 - Implement a self-join on the customer dataset to identify hierarchical relationships

In [209]:
import pandas as pd
import numpy as np

# Create a sample Employees DataFrame
employees_df = pd.DataFrame({
    'EmployeeID': [1, 2, 3, 4, 5],
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'ManagerID': [np.nan, 1, 1, 2, 2] # np.nan for the CEO/top-level manager
})

print("Original Employees DataFrame:")
print(employees_df)

# Perform a self-join to link employees to their managers
# 'left_on' refers to a column in the left DataFrame (employees_df, seen as employees)
# 'right_on' refers to a column in the right DataFrame (employees_df, seen as managers)
# 'suffixes' helps distinguish columns with the same name (e.g., 'Name')
self_joined_df = pd.merge(
    employees_df,           # Left DataFrame (aliased as 'employees')
    employees_df,           # Right DataFrame (aliased as 'managers')
    left_on='ManagerID',    # Column in 'employees' that matches 'EmployeeID' in 'managers'
    right_on='EmployeeID',  # Column in 'managers' that matches 'ManagerID' in 'employees'
    how='left',             # Use a left join to include all employees, even those without a manager (like Alice)
    suffixes=('_employee', '_manager') # Add suffixes to differentiate 'Name' and 'EmployeeID' columns
)

# Select and rename relevant columns for a cleaner output
result_df = self_joined_df[[
    'EmployeeID_employee', 'Name_employee', 'Name_manager'
]].rename(columns={
    'EmployeeID_employee': 'Employee ID',
    'Name_employee': 'Employee Name',
    'Name_manager': 'Manager Name'
})

print("\nSelf-Joined DataFrame (Employee and their Manager):")
print(result_df)

Original Employees DataFrame:
   EmployeeID     Name  ManagerID
0           1    Alice        NaN
1           2      Bob        1.0
2           3  Charlie        1.0
3           4    David        2.0
4           5      Eve        2.0

Self-Joined DataFrame (Employee and their Manager):
   Employee ID Employee Name Manager Name
0            1         Alice          NaN
1            2           Bob        Alice
2            3       Charlie        Alice
3            4         David          Bob
4            5           Eve          Bob


2. **Handling Duplicates and Conflicts**
   - Identify and handle duplicate keys when joining datasets

In [213]:
transactions_no_duplicates = transactions_df.drop_duplicates(subset=['transaction_id'])
transactions_no_duplicates.reset_index(drop=True, inplace=True)
#transactions_df.shape()
#transactions_no_duplicates.shape()
print("transactions with duplicates:")
print(transactions_df.describe())
print("transactions without duplicates:")  
print(transactions_no_duplicates.describe())



transactions with duplicates:
                                date       quantity     unit_price  \
count                         150805  150805.000000  149304.000000   
mean   2021-12-30 23:00:24.373197056       2.499566     523.856358   
min              2020-01-01 00:00:00       1.000000      18.690000   
25%              2021-01-01 00:00:00       1.000000     256.330000   
50%              2021-12-31 00:00:00       3.000000     542.160000   
75%              2022-12-29 00:00:00       3.000000     785.330000   
max              2023-12-30 00:00:00       4.000000     995.680000   
std                              NaN       1.117892     295.629985   

        discount_pct  price_after_discount   total_amount       store_id  
count  149303.000000         150805.000000  150805.000000  150805.000000  
mean        0.025160            510.481820    1268.809646      24.968482  
min         0.000000             14.950000      -1.000000       1.000000  
25%         0.000000            246.760

   - Implement a custom conflict resolution strategy when merging data with overlapping columns

In [214]:
def handle_conflicts_merging(df1, df2, on, how='inner'):
    """
    Merges two DataFrames with conflict resolution.
    
    Parameters:
    - df1: First DataFrame
    - df2: Second DataFrame
    - on: Column(s) to join on
    - how: Type of join ('inner', 'outer', 'left', 'right')
    
    Returns:
    - Merged DataFrame with conflicts resolved
    """
    merged_df = pd.merge(df1, df2, on=on, how=how, indicator=True)
    
    # Handle conflicts by keeping the first occurrence
    merged_df = merged_df.drop_duplicates(subset=on)
    
    return merged_df

   - Create a function that validates the integrity of joined data

In [216]:
def validate_and_clean_data(df, required_columns):
    """
    Validates and cleans a DataFrame.
    
    Parameters:
    - df: DataFrame to validate
    - required_columns: List of required columns
    
    Returns:
    - Cleaned DataFrame or None if validation fails
    """
    # Check for missing required columns
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        print(f"Missing required columns: {missing_cols}")
        return None
    
    # Drop rows with any NaN values in the required columns
    cleaned_df = df.dropna(subset=required_columns)
    
    return cleaned_df.reset_index(drop=True)

3. **Time-Based Joins**

   - Perform asof joins to match records based on timestamps

- For time-based joins, make sure to sort your data by time first
- The `pd.merge_asof` function requires sorted data on the join keys

In [219]:
def merge_with_asof(df1,df2):
    """
    Merges two DataFrames using an asof merge.
    
    Parameters:
    - df1: First DataFrame
    - df2: Second DataFrame
    
    Returns:
    - Merged DataFrame
    """
    # Ensure both DataFrames are sorted by the merge key
    df1 = df1.sort_values('date')
    df2 = df2.sort_values('date')
    
    # Perform the asof merge
    merged_df = pd.merge_asof(df1, df2, on='date', direction='backward')
    
    return merged_df.reset_index(drop=True)

   - Implement a rolling join that matches each transaction with the most recent customer status update

In [ ]:
def rolling_window_latest_customer_update(transaction_df, status_updates_df):
    """
    Applies a rolling window to get the latest customer updates.
    
    Parameters:
    - df: DataFrame with customer updates
    - window_size: Size of the rolling window (default is 30 days)
    
    Returns:
    - DataFrame with latest updates within the rolling window
    """
    df['update_date'] = pd.to_datetime(df['update_date'])
    df.set_index('update_date', inplace=True)
    
    # Apply rolling window
    latest_updates = df.rolling(window=window_size).last().reset_index()
    
    return latest_updates

   - Create a time window join to match events that occurred within a specified time range of each other

In [220]:
def time_window(df1, df2):
    """
    Merges two DataFrames based on a time window.
    
    Parameters:
    - df1: First DataFrame
    - df2: Second DataFrame
    
    Returns:
    - Merged DataFrame with time window applied
    """
    # Ensure both DataFrames are sorted by the merge key
    df1 = df1.sort_values('date')
    df2 = df2.sort_values('date')
    
    # Perform the merge with a time window
    merged_df = pd.merge_asof(df1, df2, on='date', direction='backward')
    
    return merged_df.reset_index(drop=True)

### Task 3: Performance Optimization

### Task 3 Hints:

- Use `pd.to_numeric()` with `downcast` parameter to reduce memory usage
- Consider using `pd.Categorical` for columns with few unique values
- For large datasets, implement a generator that yields chunks of the data
- Vectorized string operations can be performed using `str` accessor methods

1. **Memory Usage Optimization**


 - Analyze memory usage of the datasets using `memory_usage(deep=True)`

In [221]:
# memory_usage = pd.DataFrame({
#     'DataFrame': ['customers_df', 'products_df', 'transactions_df', 'status_updates_df'],
#     'Memory_Usage_MB': [  

   - Optimize datatypes to reduce memory footprint (e.g., using categories, smaller integer types)

In [222]:
def reduce_memory_usage(df):
    """
    Reduces memory usage of a DataFrame by downcasting numeric types.
    
    Parameters:
    - df: DataFrame to reduce memory usage
    
    Returns:
    - DataFrame with reduced memory usage
    """
    for col in df.select_dtypes(include=['float']):
        df[col] = pd.to_numeric(df[col], downcast='float')
    
    for col in df.select_dtypes(include=['int']):
        df[col] = pd.to_numeric(df[col], downcast='integer')
    
    return df

   - Implement chunking strategies for processing large datasets

In [223]:
def chunk_data(df, chunk_size):
    """
    Splits a DataFrame into smaller chunks.
    
    Parameters:
    - df: DataFrame to split
    - chunk_size: Size of each chunk
    
    Returns:
    - List of DataFrame chunks
    """
    return [df[i:i + chunk_size] for i in range(0, df.shape[0], chunk_size)]

2. **Computational Efficiency**

   - Compare the performance of vectorized operations versus apply/iterrows

In [224]:
# Compare the performance of vectorized operations versus apply/iterators


   - Implement an efficient strategy for updating values based on complex conditions

In [225]:
#Implement an efficient strategy for updating values based on complex conditions

   - Use `numba` or `swifter` to accelerate pandas operations

In [226]:
#   - Use `numba` or `swifter` to accelerate pandas operations

3. **Parallel Processing**

   - Split the data into chunks and process in parallel using multiprocessing

In [227]:
#   - Split the data into chunks and process in parallel using multiprocessing

   - Implement parallel group operations using `dask` or `modin`

In [228]:
#   - Implement parallel group operations using `dask` or `modin`

   - Compare execution times between sequential and parallel approaches

In [229]:
#   - Compare execution times between sequential and parallel approaches

### Task 4: Custom Extensions and Advanced Functionality

### Task 4 Hints:

- Look into `pandas.api.extensions.register_dataframe_accessor` and `pandas.api.extensions.register_series_accessor`
- For method chaining, ensure your custom methods return a DataFrame or Series object
- Consider creating a class that encapsulates your ETL logic for reusability

1. **Custom Accessors**

   - Create a custom pandas accessor that adds domain-specific functionality

In [230]:
#   - Create a custom pandas accessor that adds domain-specific functionality

   - Implement methods for sales-specific calculations and transformations

In [231]:
#   - Implement methods for sales-specific calculations and transformations

   - Add property accessors that compute derived metrics

In [232]:
#   - Add property accessors that compute derived metrics

2. **Method Chaining and Pipelines**

   - Create a data processing pipeline using method chaining

In [233]:
#   - Create a data processing pipeline using method chaining

   - Implement custom methods that preserve the DataFrame interface

In [234]:
#   - Implement custom methods that preserve the DataFrame interface

   - Design a reusable ETL process for the sales data

In [235]:
#   - Design a reusable ETL process for the sales data

3. **Advanced Visualization Integration**

   - Create a custom plotting method that generates a dashboard of sales metrics

In [236]:
#   - Create a custom plotting method that generates a dashboard of sales metrics

   - Implement interactive visualizations using plotly

In [237]:
#   - Implement interactive visualizations using plotly

   - Design a function that automatically generates reports with relevant visualizations

### Task 5: Real-world Challenge

### Task 5 Hints:

- Break down the complex task into modular components
- Consider using `pd.MultiIndex` to organize multi-dimensional data
- Implement efficient data structures for the recommendation engine
- Use pandas time-series functionality (like `resample` and `rolling`) for forecasting

Implement a complete solution for a retail analytics scenario:

1. Process raw transaction data to handle missing values, duplicates, and outliers

In [238]:
#1. Process raw transaction data to handle missing values, duplicates, and outliers

2. Create customer profiles with calculated metrics (lifetime value, purchase frequency, etc.)

In [239]:
#2. Create customer profiles with calculated metrics (lifetime value, purchase frequency, etc.)

3. Develop a product recommendation engine based on co-purchase patterns

In [240]:
#3. Develop a product recommendation engine based on co-purchase patterns

4. Implement time-series forecasting for product sales

In [241]:
#4. Implement time-series forecasting for product sales

5. Create an anomaly detection system for unusual transaction patterns

In [242]:
#5. Create an anomaly detection system for unusual transaction patterns

6. Generate a comprehensive dashboard with actionable insights

In [243]:
#6. Generate a comprehensive dashboard with actionable insights